# Phase 2 — GridWorld MDP

Reinforces Sutton & Barto Ch 3 in code. Three exercises adapted from
*AI Engineering from Scratch — Phase 9 / Lesson 1* (see
`notes/phase2-rl-foundations.md`).

Cross-refs into this repo:
- [`exercises/sutton-barto-ch03.md`](../exercises/sutton-barto-ch03.md) —
  §3.14 (numerical Bellman) and §3.24 (optimal gridworld value).
- [`notes/phase2-rl-foundations.md`](../notes/phase2-rl-foundations.md) —
  §3.1 (4-arg $p$), §3.4 (discount vs horizon), §3.5 (info sets).
- [`exercises/ex01_environment.md`](../exercises/ex01_environment.md) —
  §4 (linearity of expectation).

Author fills in every `_( implement )_` block. Don't remove the cross-refs
in the markdown cells — they anchor the artifact to the rest of the
study notes.

## Setup

In [ ]:
from __future__ import annotations

import random

import matplotlib.pyplot as plt
import numpy as np

GRID = 4
TERMINAL = (GRID - 1, GRID - 1)
ACTIONS = {
    "up": (-1, 0),
    "down": (1, 0),
    "left": (0, -1),
    "right": (0, 1),
}


def all_states() -> list[tuple[int, int]]:
    """All cells of the 4x4 grid, including the terminal."""
    return [(r, c) for r in range(GRID) for c in range(GRID)]

## Step 1 — The MDP

**Task.** Implement the deterministic step function. Signature:

```
step(state: tuple[int, int], action: str) -> (next_state, reward, done)
```

Rules:
- Terminal state absorbs: `step(TERMINAL, *) == (TERMINAL, 0.0, True)`.
- Any other state: apply the action delta from `ACTIONS`, clip to the
  grid bounds, reward `-1.0`, `done` is True iff the next state is
  `TERMINAL`.

In [ ]:
def step(state: tuple[int, int], action: str) -> tuple[tuple[int, int], float, bool]:
    """Deterministic 4x4 GridWorld transition."""
    if state == TERMINAL:
        return TERMINAL, 0.0, True
    dr, dc = ACTIONS[action]
    row = min(max(state[0] + dr, 0), GRID - 1)
    col = min(max(state[1] + dc, 0), GRID - 1)
    nxt = (row, col)
    return nxt, -1.0, nxt == TERMINAL

## Step 2 — Policy and rollout

**Task.** Implement `uniform_policy(state)` (returns a dict mapping each
action to $0.25$) and `rollout(policy, max_steps=200)` (returns
`(total_return, steps)`).

The rollout should sample from the policy's action distribution using
the standard-library `random.choices` — a small helper `sample_action`
is provided.

In [ ]:
def sample_action(policy_dist: dict[str, float], rng: random.Random) -> str:
    """Sample one action from a discrete distribution."""
    actions, probs = zip(*policy_dist.items())
    return rng.choices(actions, weights=probs, k=1)[0]


def uniform_policy(state: tuple[int, int]) -> dict[str, float]:
    """Uniform-random policy over the four cardinal actions."""
    return {action: 1.0 / len(ACTIONS) for action in ACTIONS}


def rollout(
    policy,
    max_steps: int = 200,
    rng: random.Random | None = None,
    start: tuple[int, int] = (0, 0),
) -> tuple[float, int]:
    """Roll out a policy from `start` until termination or max_steps.

    Returns:
        total_return: sum of undiscounted rewards along the trajectory.
        steps: number of environment steps taken.
    """
    rng = rng or random.Random()
    state = start
    total_return = 0.0
    steps = 0
    for _ in range(max_steps):
        if state == TERMINAL:
            break
        action = sample_action(policy(state), rng)
        state, reward, done = step(state, action)
        total_return += reward
        steps += 1
        if done:
            break
    return total_return, steps

## Exercise Easy — 10,000 rollouts

**Task.** Run 10,000 rollouts of the uniform-random policy. Report:

- mean total return
- standard deviation of total return
- mean episode length (steps)

Compare against the theoretical optimum. For a 4×4 grid with step
penalty $-1$, the optimal path length from `(0, 0)` to `(3, 3)` is
6 steps (three rights + three downs, any interleaving), so the
optimal return is $-6$.

**Prompt to reflect on** (write your answer below the run):
What does the gap between random and optimal tell you about the value
of *any* policy, even a trivial one, over uniform random?

In [ ]:
N_ROLLOUTS = 10_000
MAX_STEPS = 200
OPTIMAL_RETURN = -6.0

rng = random.Random(20260716)
returns, lengths = zip(
    *(rollout(uniform_policy, max_steps=MAX_STEPS, rng=rng) for _ in range(N_ROLLOUTS))
)
returns = np.array(returns)
lengths = np.array(lengths)

print(f"rollouts          : {N_ROLLOUTS}")
print(f"mean return       : {returns.mean():8.2f}")
print(f"std  return       : {returns.std(ddof=1):8.2f}")
print(f"mean steps        : {lengths.mean():8.2f}")
print(f"optimal return    : {OPTIMAL_RETURN:8.2f}")
print(f"gap (random - opt): {returns.mean() - OPTIMAL_RETURN:8.2f}")
print(f"cost ratio        : {returns.mean() / OPTIMAL_RETURN:8.2f}x the optimal cost")

# The truncation is not cosmetic: an episode cut at max_steps is recorded as
# -200 when its true return is worse than that, so every statistic above is
# optimistic. Report the censoring rate alongside the mean, never without.
truncated = int((lengths >= MAX_STEPS).sum())
print(
    f"\ncensored at {MAX_STEPS} steps: {truncated}/{N_ROLLOUTS} "
    f"({truncated / N_ROLLOUTS:.2%}) — these bias mean/std toward zero"
)

**Reflection.** The uniform-random policy pays several times the optimal
cost of $-6$ on a grid where the goal is six steps away, and the standard
deviation is of the same order as the mean — the random walk sometimes
stumbles in quickly and sometimes wanders for a hundred steps. Two things
follow.

First, **almost the entire gap is available to trivial structure, not to
optimality**. The random walk is bad not because it plays the endgame
imprecisely but because it spends most of its steps undoing itself: it has
no bias toward the goal at all. A policy that merely prefers *right* and
*down* — no search, no values, no learning — collects most of the gap
immediately. Optimality is the last slice, and it is the expensive one.

Second, the variance is the real lesson for this project. Estimating the
value of the random policy to within a step takes thousands of episodes,
because each sample is nearly uninformative. That is exactly the cost
structure MCTS pays with random rollouts: the estimate at a node is an
average over trajectories this noisy, so the simulation budget buys
precision slowly. Biasing the rollout toward sensible play shrinks both the
gap *and* the variance of every sample used to estimate it — which is why
[ADR-001](../docs/adr/adr-001-why-ismcts.md) schedules heuristic-guided
rollouts for Phase 4 rather than keeping the Phase 3 random ones. This
notebook is the smallest setting where that argument is checkable rather
than merely plausible.

Note the censoring line in the output: episodes cut at 200 steps are
recorded as $-200$ when their true return is worse, so the mean above is
optimistic. The bias is small here, but the habit is the point — a
truncated tail is a *result*, not a rounding detail (the same reasoning
that made EXP-007's TIMEOUTs a finding rather than a bug).

## Step 3 — Iterative policy evaluation

**Task.** Implement iterative policy evaluation for the Bellman
expectation equation:

$$
V^\pi(s) \;=\; \sum_a \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a) \bigl[ r + \gamma\, V^\pi(s') \bigr].
$$

For a deterministic MDP, $p(s', r \mid s, a) = 1$ for the unique
$(s', r)$ pair returned by `step`, so the inner sum collapses to a
single term.

Loop over states until the max update magnitude falls below `tol`.

In [ ]:
def policy_evaluation(
    policy,
    gamma: float = 0.99,
    tol: float = 1e-6,
    max_iters: int = 10_000,
) -> dict[tuple[int, int], float]:
    """Iterative policy evaluation for a deterministic MDP.

    Sweeps in place (Gauss-Seidel): updates use the newest available
    V[s'] rather than a frozen copy, which converges in fewer sweeps and
    to the same fixed point.

    V[TERMINAL] stays 0 by construction — the terminal absorbs with
    reward 0, so its Bellman backup is identically zero and it is skipped.
    """
    V = {s: 0.0 for s in all_states()}
    for _ in range(max_iters):
        delta = 0.0
        for s in all_states():
            if s == TERMINAL:
                continue
            v_new = 0.0
            for action, pi_a in policy(s).items():
                # Deterministic MDP: p(s', r | s, a) = 1 for the single
                # (s', r) that step() returns, so the inner sum collapses.
                s_next, reward, _ = step(s, action)
                v_new += pi_a * (reward + gamma * V[s_next])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < tol:
            break
    return V


def v_to_grid(V: dict[tuple[int, int], float]) -> np.ndarray:
    """Convert V (dict) into a (GRID, GRID) numpy array for display."""
    grid = np.zeros((GRID, GRID))
    for (r, c), v in V.items():
        grid[r, c] = v
    return grid

## Exercise Medium — γ ∈ {0.5, 0.9, 0.99}

**Task.** Run `policy_evaluation(uniform_policy, gamma=g)` for the
three values of $\gamma$. Print $V$ as a 4×4 grid (numeric or heatmap
via `matplotlib.pyplot.imshow`) for each.

**Prompt to reflect on** (write your answer below):
Why do state values near the terminal grow faster with larger $\gamma$?
Connect this to the effective-horizon interpretation
$\text{H}_{\text{eff}} \approx 1 / (1 - \gamma)$ from
[`notes/phase2-rl-foundations.md`](../notes/phase2-rl-foundations.md) §3.4.

In [ ]:
GAMMAS = [0.5, 0.9, 0.99]
V_by_gamma = {g: policy_evaluation(uniform_policy, gamma=g) for g in GAMMAS}

fig, axes = plt.subplots(1, len(GAMMAS), figsize=(13, 4))
for ax, g in zip(axes, GAMMAS):
    grid = v_to_grid(V_by_gamma[g])
    im = ax.imshow(grid, cmap="viridis")
    ax.set_title(rf"$\gamma$ = {g}   ($H_{{eff}} \approx$ {1 / (1 - g):.0f} steps)")
    ax.set_xticks(range(GRID))
    ax.set_yticks(range(GRID))
    for r in range(GRID):
        for c in range(GRID):
            ax.text(c, r, f"{grid[r, c]:.2f}", ha="center", va="center",
                    color="w", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(r"$V^\pi$ under the uniform-random policy")
plt.tight_layout()
plt.show()

# Each colorbar is on its own scale — the heatmaps show the *shape* of V,
# not its magnitude. The numbers below are what the reflection argues from.
print(f"{'gamma':>6} {'H_eff':>7} {'V[start]':>10} {'V[(2,3)]':>10} "
      f"{'floor -1/(1-g)':>15} {'spread':>8}")
for g in GAMMAS:
    V = V_by_gamma[g]
    values = [v for s, v in V.items() if s != TERMINAL]
    print(f"{g:>6} {1 / (1 - g):>7.0f} {V[(0, 0)]:>10.3f} {V[(2, 3)]:>10.3f} "
          f"{-1 / (1 - g):>15.3f} {max(values) - min(values):>8.3f}")

**Reflection.** First, a precision about the question. Raising $\gamma$
makes *every* value more negative, not more positive — the reward is $-1$
per step and a patient agent counts more of them. What grows is the
**spread**: how far near-terminal states pull away from the rest. The
printed table makes this checkable.

The mechanism is the geometric floor. With every reward equal to $-1$,

$$V^\pi(s) \;\ge\; -\sum_{k=0}^{\infty} \gamma^k \;=\; -\frac{1}{1 - \gamma},$$

which is $-2$ at $\gamma = 0.5$, $-10$ at $\gamma = 0.9$, $-100$ at
$\gamma = 0.99$. A state only rises above that floor to the extent it
expects to *stop paying* — i.e. to the extent it can reach the terminal.
Reaching it $k$ steps from now is worth $\gamma^k$, so the terminal is
invisible from more than roughly $H_{\text{eff}} \approx 1/(1-\gamma)$
steps away
([`notes/phase2-rl-foundations.md`](../notes/phase2-rl-foundations.md)
§3.4).

That is what the three panels show. At $\gamma = 0.5$, $H_{\text{eff}}
\approx 2$: only the cells within a step or two of $(3,3)$ lift off the
$-2$ floor, and the rest of the grid saturates into a nearly flat sheet.
The geometry is *invisible* — $V$ cannot represent "the goal is six steps
away" because six steps are worth $0.5^6 \approx 1.6\%$. At $\gamma =
0.99$, $H_{\text{eff}} \approx 100$ comfortably exceeds both the 6-step
optimal path and the ~20-step random hitting time, so the gap to the floor
is large everywhere and orders the cells by true expected cost-to-go. The
near-terminal states grow fastest *relative to the floor* because they are
the ones with something to discount back.

The iterative reading is the same fact in motion: each sweep propagates
information one cell outward from the terminal, and $\gamma^k$ decides
whether it survives the trip. Low $\gamma$ is not "impatience" as a
preference — it is a *representational* limit on how far the value
function can see.

The consequence for this project is why we use $\gamma = 1$ with a terminal
reward and no shaping
([ADR-004](../docs/adr/adr-004-terminal-reward-not-shaped.md)): a PTCG game
is tens of decisions long and the only reward that matters — winning —
arrives at the end. Any $\gamma$ whose $H_{\text{eff}}$ is shorter than the
game would discount the win into noise, and we would be optimizing a
horizon we did not intend to choose.

## Step 4 — Stochastic step (slip)

**Task.** Implement a stochastic variant of `step` where the intended
action succeeds with probability $1 - p_{\text{slip}}$, and with
probability $p_{\text{slip}}$ the agent instead moves in one of the
other three cardinal directions (uniformly among them, so each
non-intended direction has probability $p_{\text{slip}} / 3$).

The transition kernel now has $|s'| = 4$ possible successors per
`(state, action)` pair, in contrast to the deterministic version's
$|s'| = 1$. Update `policy_evaluation_stochastic` accordingly — it
must marginalize over successors:

$$
V^\pi(s) \;=\; \sum_a \pi(a \mid s) \sum_{s'} p(s' \mid s, a)\, \bigl[ r(s, a, s') + \gamma\, V^\pi(s') \bigr].
$$

In [ ]:
P_SLIP = 0.1


def slip_kernel(intended: str, p_slip: float = P_SLIP) -> dict[str, float]:
    """p(effective action | intended action) under slip.

    The intended action survives w.p. (1 - p_slip); the remaining mass is
    split evenly over the other three, so each gets p_slip / 3.

    This kernel is *doubly stochastic*: every row sums to 1 by
    construction, and so does every column — (1 - p_slip) + 3 * (p_slip/3)
    = 1. That property is the whole point of the Hard exercise below.
    """
    others = [a for a in ACTIONS if a != intended]
    dist = {intended: 1.0 - p_slip}
    dist.update({a: p_slip / len(others) for a in others})
    return dist


def step_stochastic(
    state: tuple[int, int],
    action: str,
    rng: random.Random,
    p_slip: float = P_SLIP,
) -> tuple[tuple[int, int], float, bool]:
    """Stochastic step: intended action w.p. (1 - p_slip), else uniform slip."""
    if state == TERMINAL:
        return TERMINAL, 0.0, True
    effective = sample_action(slip_kernel(action, p_slip), rng)
    # Bounds clipping, reward and done are unchanged — only which action
    # actually gets executed is random, so delegate to the deterministic step.
    return step(state, effective)


def policy_evaluation_stochastic(
    policy,
    gamma: float = 0.99,
    tol: float = 1e-6,
    p_slip: float = P_SLIP,
    max_iters: int = 10_000,
) -> dict[tuple[int, int], float]:
    """Iterative policy evaluation for the slippery GridWorld.

    Enumerates all four possible successor states per (s, a) instead of
    following a single sampled transition.
    """
    V = {s: 0.0 for s in all_states()}
    for _ in range(max_iters):
        delta = 0.0
        for s in all_states():
            if s == TERMINAL:
                continue
            v_new = 0.0
            for action, pi_a in policy(s).items():
                # Marginalize over the four possible effective actions:
                #   sum_{s'} p(s' | s, a) [ r(s, a, s') + gamma V(s') ].
                # Distinct effective actions can clip to the same s' at a
                # wall; summing (rather than assigning) keeps that correct.
                for effective, p_trans in slip_kernel(action, p_slip).items():
                    s_next, reward, _ = step(s, effective)
                    v_new += pi_a * p_trans * (reward + gamma * V[s_next])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < tol:
            break
    return V

## Exercise Hard — V under slip

**Task.** Run `policy_evaluation_stochastic(uniform_policy, gamma=0.99,
p_slip=0.1)`. Print the resulting $V$ grid and compare it side-by-side
with the deterministic version from Exercise Medium (with the same
$\gamma$).

**Prompt to reflect on** (write your answer below):
Does $V[\text{start}]$ (i.e. $V[(0, 0)]$) get better or worse under
slip, holding $\gamma$ fixed? Why?

Hint: think about how a uniform-random policy is affected by transition
noise. Under uniform-random *and* deterministic transitions, the agent
already wanders. Adding slip introduces additional wandering on top of
an already-random policy. Which direction should $V[\text{start}]$
move?

In [ ]:
GAMMA = 0.99
V_det = V_by_gamma[GAMMA]  # reuse Exercise Medium, same gamma
V_slip = policy_evaluation_stochastic(uniform_policy, gamma=GAMMA, p_slip=P_SLIP)

diff = v_to_grid(V_slip) - v_to_grid(V_det)
panels = [
    ("deterministic", v_to_grid(V_det), "viridis"),
    (rf"slip $p$ = {P_SLIP}", v_to_grid(V_slip), "viridis"),
    ("slip - deterministic", diff, "coolwarm"),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, grid, cmap) in zip(axes, panels):
    im = ax.imshow(grid, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(range(GRID))
    ax.set_yticks(range(GRID))
    for r in range(GRID):
        for c in range(GRID):
            ax.text(c, r, f"{grid[r, c]:.2f}", ha="center", va="center",
                    color="k", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(rf"Uniform-random policy, $\gamma$ = {GAMMA}")
plt.tight_layout()
plt.show()

max_abs_diff = float(np.abs(diff).max())
print("--- uniform-random policy ---")
print(f"V[start] deterministic : {V_det[(0, 0)]:.6f}")
print(f"V[start] slip p={P_SLIP}    : {V_slip[(0, 0)]:.6f}")
print(f"max |V_slip - V_det|   : {max_abs_diff:.3e}   "
      f"(tol of the sweep is 1e-06)")

# Control: slip is only invisible because the policy is uniform. Give the
# policy an actual intention and the same noise starts costing real return.
def rightdown_policy(state: tuple[int, int]) -> dict[str, float]:
    """A trivially non-uniform policy: only right and down, 50/50."""
    return {"right": 0.5, "down": 0.5}


V_rd_det = policy_evaluation(rightdown_policy, gamma=GAMMA)
V_rd_slip = policy_evaluation_stochastic(rightdown_policy, gamma=GAMMA,
                                         p_slip=P_SLIP)
print("\n--- right/down policy (same grid, same gamma, same slip) ---")
print(f"V[start] deterministic : {V_rd_det[(0, 0)]:.6f}")
print(f"V[start] slip p={P_SLIP}    : {V_rd_slip[(0, 0)]:.6f}")
print(f"cost of slip           : {V_rd_slip[(0, 0)] - V_rd_det[(0, 0)]:.6f}")

**Reflection.** Neither. $V[(0,0)]$ does not move — the two value functions
are *identical*, and the printed `max |V_slip - V_det|` sits at the sweep
tolerance rather than anywhere near the scale of $V$ itself. The hint's
intuition ("slip adds wandering on top of an already-random policy, so it
must get worse") is wrong, and it is worth seeing exactly where it fails.

Write the stochastic backup and exchange the two sums:

$$
V^\pi(s) = \sum_a \pi(a \mid s) \sum_{a'} K(a' \mid a)\,\bigl[r + \gamma V^\pi(s'(s,a'))\bigr]
         = \sum_{a'} \underbrace{\Bigl[\sum_a \pi(a \mid s) K(a' \mid a)\Bigr]}_{\textstyle \mu(a' \mid s)} \bigl[r + \gamma V^\pi(s'(s,a'))\bigr].
$$

Only the **induced** action distribution $\mu$ enters — the agent's
intention and the slip noise reach the environment through their
composition and nothing else. For the slip kernel,
$K(a' \mid a) = (1 - p)$ if $a' = a$ and $p/3$ otherwise, so with
$\pi = \tfrac14$ uniform:

$$
\mu(a' \mid s) = \tfrac14 \sum_a K(a' \mid a) = \tfrac14\Bigl[(1 - p) + 3 \cdot \tfrac{p}{3}\Bigr] = \tfrac14 .
$$

$\mu$ *is* $\pi$. The stochastic Bellman operator is literally the
deterministic one, so — being a $\gamma$-contraction with a unique fixed
point — it converges to the same $V$. The equality is exact, not
approximate, and holds for every $p_{\text{slip}} \in [0, 1]$.

The structural reason is that $K$ is **doubly stochastic** (its columns sum
to $1$ as well as its rows), and the uniform distribution is invariant
under any such kernel. Noise destroys *information*, and a uniform policy
has none to destroy: it is already at maximum entropy over the four
actions, so there is nothing left to randomize. You cannot make a coin
flip more random by flipping a second coin about it.

The `rightdown_policy` control is the other half of the claim. Give the
policy the barest intention — 50/50 over two of the four actions, still no
search, no values — and the *same* slip kernel now costs real return,
because now there is something for the noise to corrupt. **Transition
stochasticity is a tax on intention, and it scales with how much intention
you have.**

Two guards against over-reading this. The invariance is a property of
$V^\pi$ for the uniform $\pi$, not of the MDP: $V^*$ and the optimal policy
*do* degrade under slip. And it depends on $K$ being doubly stochastic — an
asymmetric slip (say, always drifting right) would break the column sum,
induce a non-uniform $\mu$, and move $V$ even for the uniform policy.

This is the sharpest thing in the notebook for the PTCG project, and it
inverts a comfortable assumption. Coin flips and random draws hurt a
*strong* agent more than a weak one, so the better our search gets, the
more the game's stochasticity costs us — which is precisely why
[ADR-001](../docs/adr/adr-001-why-ismcts.md) flags that chance-node
determinization inflates the variance of the value estimate and demands
more simulations for a stable UCB1 signal. It also carries a benchmarking
warning: measuring against a random opponent systematically *understates*
how much variance matters, because a random opponent is the one player in
the game that noise cannot hurt.

## Bridge to the PTCG project

**Task.** Fill in the following comparison table with one-line
observations from what you built above. This is where GridWorld
earns its keep as a study artifact — the table below is what
distinguishes "did an RL exercise" from "understood *why* PTCG needs
a different algorithm."

| Dimension | GridWorld (this notebook) | Pokémon TCG (this project) | Consequence for algorithm choice |
|---|---|---|---|
| Observability | Fully observed (state = cell) | Imperfect: the opponent's hand and both decks are hidden; we see an information set $I_i(s)$, not $s$ ([`ex01`](../exercises/ex01_environment.md) §2) | Every backup here needs $s$; we never have one. Forces determinization — sample $h \sim P(h \mid I)$, search that world, marginalize over samples ([ADR-001](../docs/adr/adr-001-why-ismcts.md)) |
| Transition determinism | Deterministic: `step` returns one successor, so $p(s', r \mid s, a) = 1$ and the inner Bellman sum collapses | Stochastic (coin flips, card draws) | Slip gives 4 successors — still few enough to *enumerate* (`policy_evaluation_stochastic` is an expected update). PTCG's fan-out is not, so MCTS is pure **sample** updates ([`phase2-rl-foundations`](../notes/phase2-rl-foundations.md) §8.3) |
| Reward density | Dense: $-1$ on every step, so every transition teaches something | Sparse: only the terminal outcome, $r_T \in \{-1, 0, +1\}$ at game end ([ADR-004](../docs/adr/adr-004-terminal-reward-not-shaped.md)) | No per-step signal to bootstrap from → each simulation must run to a terminal before it returns one bit, and credit for a turn-3 mistake arrives tens of decisions later |
| State-space size | $16$ cells — $V$ is a dict, and a sweep enumerates all of $S$ | $\lvert I_i(s) \rvert \lesssim 10^{50}$ mid-game ([`ex01`](../exercises/ex01_environment.md) §3) | Tabular anything is out, and so is any sweep over $S$. The tree must be built lazily from the root and only where the search actually visits |
| Right family of algorithms | Value iteration, Q-learning | SO-ISMCTS: one information-set tree, per-iteration uniform determinization, UCB1 selection ([ADR-001](../docs/adr/adr-001-why-ismcts.md)) | Anytime (matters — EXP-008's whole budget policy rests on it), needs no model sweep, and handles imperfect information without enumerating a belief state |

The two `policy_evaluation` variants above are worth naming for what they
are: the deterministic one is an *expected* update over a single successor,
the slippery one an expected update over four. Both enumerate. The
`rollout` function is the *sample* update — one trajectory, no enumeration.
GridWorld is the last environment in this project where we get to choose;
past a branching factor of a few dozen the choice is made for us, which is
the argument §8.3 makes and the reason every line of `search/ismcts.py`
looks the way it does.

**Final reflection.** _( 3-5 lines: what did coding this shift in your
understanding vs just reading Ch 3? )_